# Model Skor Risiko Pembatalan Pesanan
Febrianti, Data Analyst / ML Engineer

Alur: muat `fact_orders`, buat label dan saring baris, pilih fitur aman, split berdasarkan waktu, latih baseline dan Boosted Decision Tree, evaluasi (ROC-AUC, PR-AUC, Recall), hitung skor 0-100, simpan hasil.

Letakkan notebook ini di folder yang sama dengan `fact_orders.parquet` (atau `fact_orders.csv`). Paket: pandas, numpy, scikit-learn, pyarrow.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score, precision_score,
                             f1_score, accuracy_score, confusion_matrix, precision_recall_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
PARQUET = Path("../../data/gold/fact_orders.parquet")
CSV = Path("../../data/gold/fact_orders.csv")

## 1. Muat dan periksa data
Hasil pemeriksaan yang sudah dilakukan pada `fact_orders`: 20.848 baris, 0 duplikat `order_id`, 2.830 batal, label dan waktu cocok dengan data mentah, 4 kolom bocor dan `Opsi Pengiriman` sudah dihapus, `lat`/`long` tidak ada yang kosong, `is_ramadan` berisi 951 pesanan.

In [2]:
df = pd.read_csv(CSV)
df["waktu"] = pd.to_datetime(df["waktu_pesanan_dibuat"], format="mixed")
assert len(df) == 20848 and df.order_id.is_unique and df.is_batal.sum() == 2830
print(df.shape, f"| batal {df.is_batal.mean():.2%}")

(20848, 36) | batal 13.57%


## 2. Label dan penyaringan baris
- Label: `is_batal` (sudah ada, dibuat dari Status Pesanan).
- **Keluarkan 765 pesanan `nama_ekspedisi = "Belum ditetapkan"`**. Kurir kosong adalah akibat pembatalan (100% batal), bukan sebab.
- **Keluarkan 250 pesanan belum final** (sedang dikirim, telah dikirim, masa pengembalian). Hasil akhirnya belum diketahui.

In [3]:
final = df.status_pesanan.isin(["Selesai", "Batal"])
ada_kurir = df.nama_ekspedisi != "Belum ditetapkan"
print("tanpa kurir:", (~ada_kurir).sum(), "| belum final:", (~final).sum(), "| keduanya:", ((~ada_kurir) & (~final)).sum())
data = df[final & ada_kurir].copy()
print(f"dipakai {len(data)} dari {len(df)} baris | tingkat batal {data.is_batal.mean():.2%}")

tanpa kurir: 765 | belum final: 250 | keduanya: 0
dipakai 19833 dari 20848 baris | tingkat batal 10.41%


## 3. Fitur
Hanya informasi yang diketahui **sebelum pesanan dikirim**. Status Pesanan dan Alasan Pembatalan hanya dipakai untuk label dan penyaringan, bukan fitur. Kolom bocor lain (Total Pembayaran, ongkir dibayar pembeli, estimasi potongan, total_returned_qty) sudah dihapus pipeline.

In [4]:
data["jam"] = data.waktu.dt.hour
data["hari_minggu"] = data.waktu.dt.dayofweek
# kota dengan banyak kasus dipertahankan, sisanya digabung agar tidak terlalu jarang
top_kota = data.kota_kabupaten_clean.value_counts().head(100).index
data["kota"] = data.kota_kabupaten_clean.where(data.kota_kabupaten_clean.isin(top_kota), "lainnya")
# product_categories punya ratusan kombinasi; simpan 50 teratas, sisanya digabung
top_kat = data.product_categories.value_counts().head(50).index
data["kategori_produk"] = data.product_categories.where(data.product_categories.isin(top_kat), "lainnya")

KAT_DASAR = ["metode_pembayaran", "provinsi", "kota", "kategori_produk", "tipe_pengiriman"]
NUM_DASAR = ["total_qty", "total_weight_gr", "num_product_categories", "total_diskon",
             "jam", "hari_minggu", "bulan_pesanan", "is_akhir_pekan",
             "is_libur_nasional", "is_cuti_bersama", "is_event_promo", "is_ramadan", "lat", "long"]
# fitur yang masih perlu diuji: apakah sudah diketahui saat pesanan dibuat, atau bocor?
OPSI = {"perkiraan_ongkir": (["perkiraan_ongkos_kirim"], []),
        "ekspedisi":        ([], ["nama_ekspedisi"])}
TERLARANG = ["status_pesanan", "alasan_pembatalan", "is_batal", "order_id", "source_file"]
assert not set(TERLARANG) & set(KAT_DASAR + NUM_DASAR + sum([a + b for a, b in OPSI.values()], []))

C:\Users\fbrnt\AppData\Local\Temp\ipykernel_22140\774912399.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  data["hari_minggu"] = data.waktu.dt.dayofweek


## 4. Split berdasarkan waktu
Model dilatih pada masa lalu dan diuji pada masa depan. Validasi dipakai untuk memilih ambang keputusan. Tingkat batal turun di akhir 2025, jadi hasil uji bisa berbeda dari validasi.

In [5]:
TRAIN_END, VAL_END = "2025-02-28", "2025-05-31"
train = data[data.waktu <= TRAIN_END]
val   = data[(data.waktu > TRAIN_END) & (data.waktu <= VAL_END)]
test  = data[data.waktu > VAL_END]
for nama, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{nama:5s} {len(d):6d} baris | batal {d.is_batal.mean():.2%} | {d.waktu.min().date()} s/d {d.waktu.max().date()}")

train  12224 baris | batal 10.27% | 2023-12-01 s/d 2025-02-27
val     2188 baris | batal 15.36% | 2025-02-28 s/d 2025-05-30
test    5421 baris | batal 8.74% | 2025-05-31 s/d 2025-11-30


## 5. Fungsi latih dan evaluasi
- Baseline: Logistic Regression.
- Model utama: Boosted Decision Tree (`HistGradientBoostingClassifier` bawaan scikit-learn). LightGBM atau XGBoost bisa menggantikannya.
- Kelas timpang (sekitar 10% batal), jadi metrik utama ROC-AUC, PR-AUC, dan Recall. Accuracy dan F1 dicatat karena masih ada di charter.

In [6]:
def model_baseline(kat, num):
    prep = ColumnTransformer([("kat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), kat),
                              ("num", StandardScaler(), num)])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])

def model_boosting(kat, num):
    prep = ColumnTransformer([("kat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), kat),
                              ("num", "passthrough", num)])
    clf = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_depth=6, min_samples_leaf=40, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15,
        categorical_features=list(range(len(kat))), random_state=SEED)
    return Pipeline([("prep", prep), ("clf", clf)])

def pilih_ambang(y, p):
    pr, rc, th = precision_recall_curve(y, p)                     # ambang dengan F1 tertinggi di data validasi
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-9, None)
    return float(th[np.argmax(f1)])

def evaluasi(y, p, ambang):
    yhat = (p >= ambang).astype(int)
    return {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "Recall": recall_score(y, yhat), "Precision": precision_score(y, yhat, zero_division=0),
            "F1": f1_score(y, yhat), "Accuracy": accuracy_score(y, yhat), "Ambang": ambang}

def jalankan(builder, kat, num):
    cols = kat + num
    m = builder(kat, num).fit(train[cols], train.is_batal)
    amb = pilih_ambang(val.is_batal, m.predict_proba(val[cols])[:, 1])
    return m, evaluasi(test.is_batal, m.predict_proba(test[cols])[:, 1], amb)

## 6. Baseline vs Boosted Decision Tree, dengan dan tanpa fitur yang masih perlu diuji
Kalau skor naik tajam setelah memasukkan `perkiraan_ongkos_kirim` atau `nama_ekspedisi`, curigai kebocoran atau fitur yang belum diketahui saat pesanan dibuat.

In [7]:
skenario = {"Dasar": ([], []),
            "+ Perkiraan ongkir": OPSI["perkiraan_ongkir"],
            "+ Ekspedisi": OPSI["ekspedisi"],
            "+ Ongkir + Ekspedisi": (OPSI["perkiraan_ongkir"][0], OPSI["ekspedisi"][1])}
hasil, modeluji = [], {}
for nama_s, (num_tambah, kat_tambah) in skenario.items():
    kat, num = KAT_DASAR + kat_tambah, NUM_DASAR + num_tambah
    for nama_m, builder in [("Logistic Regression", model_baseline), ("Boosted Tree", model_boosting)]:
        m, h = jalankan(builder, kat, num)
        h["Model"] = f"{nama_m} | {nama_s}"
        hasil.append(h); modeluji[h["Model"]] = (m, kat + num)
tabel = pd.DataFrame(hasil).set_index("Model")[["ROC-AUC", "PR-AUC", "Recall", "Precision", "F1", "Accuracy", "Ambang"]]
print("Tingkat batal data uji (acuan PR-AUC acak):", round(test.is_batal.mean(), 4))
tabel.round(3)

Tingkat batal data uji (acuan PR-AUC acak): 0.0874


,ROC-AUC,PR-AUC,Recall,Precision,F1,Accuracy,Ambang
Model,,,,,,,
Logistic Regression | Dasar,0.609,0.133,0.403,0.121,0.186,0.692,0.575
Boosted Tree | Dasar,0.610,0.129,0.580,0.114,0.191,0.569,0.092
Logistic Regression | + Perkiraan ongkir,0.609,0.137,0.717,0.105,0.183,0.441,0.411
Boosted Tree | + Perkiraan ongkir,0.611,0.134,0.399,0.129,0.195,0.713,0.124
Logistic Regression | + Ekspedisi,0.610,0.132,0.483,0.118,0.190,0.639,0.538
Boosted Tree | + Ekspedisi,0.606,0.128,0.483,0.117,0.189,0.637,0.104
Logistic Regression | + Ongkir + Ekspedisi,0.610,0.137,0.373,0.124,0.187,0.715,0.577
Boosted Tree | + Ongkir + Ekspedisi,0.608,0.133,0.399,0.129,0.195,0.712,0.124


## 7. Model pilihan, matriks kebingungan, dan fitur terpenting
Pilih skenario dengan PR-AUC terbaik **yang masuk akal secara bisnis**, bukan yang tertinggi saja. Ubah `PILIHAN`.

In [8]:
PILIHAN = "Boosted Tree | Dasar"
m, cols = modeluji[PILIHAN]
amb = tabel.loc[PILIHAN, "Ambang"]
p = m.predict_proba(test[cols])[:, 1]
print(PILIHAN, "| ambang", round(amb, 3))
print(pd.DataFrame(confusion_matrix(test.is_batal, (p >= amb).astype(int)),
                   index=["aktual tidak batal", "aktual batal"], columns=["prediksi tidak", "prediksi batal"]))
pi = permutation_importance(m, test[cols], test.is_batal, scoring="average_precision", n_repeats=5, random_state=SEED)
pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False).head(15).round(4)

Boosted Tree | Dasar | ambang 0.092
                    prediksi tidak  prediksi batal
aktual tidak batal            2810            2137
aktual batal                   199             275


kategori_produk           0.0183
metode_pembayaran         0.0143
kota                      0.0112
provinsi                  0.0015
hari_minggu               0.0013
tipe_pengiriman           0.0013
total_weight_gr           0.0012
total_qty                 0.0012
long                      0.0005
lat                       0.0004
is_event_promo            0.0000
num_product_categories    0.0000
total_diskon              0.0000
is_ramadan                0.0000
is_akhir_pekan            0.0000
dtype: float64

## 8. Skor risiko 0-100
Skor = probabilitas batal x 100. Pesanan tanpa kurir dan belum final tidak ikut dilatih, jadi tidak diberi skor di sini. Skor pada data latih lebih optimis dari kenyataan, gunakan data uji untuk menilai kualitas.

In [9]:
skor = data[["order_id", "waktu", "provinsi", "metode_pembayaran", "nama_ekspedisi", "is_batal"]].copy()
skor["skor_risiko"] = (m.predict_proba(data[cols])[:, 1] * 100).round(1)
skor["sumber"] = np.select([data.waktu <= TRAIN_END, data.waktu <= VAL_END], ["train", "val"], "test")
skor["level_risiko"] = pd.cut(skor.skor_risiko, [-1, 20, 50, 100], labels=["Rendah", "Sedang", "Tinggi"])
print(skor[skor.sumber == "test"].groupby("level_risiko", observed=True).is_batal.agg(["size", "mean"]).round(3))
skor.to_csv("skor_risiko.csv", index=False)
print("tersimpan: skor_risiko.csv", skor.shape)

              size   mean
level_risiko             
Rendah        4936  0.081
Sedang         464  0.151
Tinggi          21  0.238
tersimpan: skor_risiko.csv (19833, 9)


## 9. Catatan untuk dokumentasi model
Catat: tanggal pembagian data, fitur yang dipakai dan dilarang beserta alasannya, hasil seluruh skenario di bagian 6, ambang yang dipilih, batasan (tingkat batal turun di akhir 2025; 765 pesanan tanpa kurir dan 250 belum final tidak dilatih), dan langkah berikutnya (tabel rekomendasi kurir dari 277 kasus kegagalan pengiriman, lalu tulis skor ke PostgreSQL).